The problem is to calculate the minimum number of platforms required at a train station based on the given arrival_times and departure_times.


#### Solution
- We need to merge both arrival_time and departure_time into a unified dataset.
- We’ll use a window function to track how many platforms are required at each point in time.
- For each train arrival, we’ll add a platform (+1) and for each train departure, we’ll subtract a platform (-1).
- Finally, we will calculate the maximum number of platforms required at any point in time during the day.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Create Spark session
spark = SparkSession.builder.appName("Train Platform Calculation").getOrCreate()
# Sample Data (Train Arrival and Departure times)
arrivals_data = [
    (1, '2024-11-17 08:00'),
    (2, '2024-11-17 08:05'),
    (3, '2024-11-17 08:05'),
    (4, '2024-11-17 08:10'),
    (5, '2024-11-17 08:10'),
    (6, '2024-11-17 12:15'),
    (7, '2024-11-17 12:20'),
    (8, '2024-11-17 12:25'),
    (9, '2024-11-17 15:00'),
    (10, '2024-11-17 15:00'),
    (11, '2024-11-17 15:00'),
    (12, '2024-11-17 15:06'),
    (13, '2024-11-17 20:00'),
    (14, '2024-11-17 20:10')
]
departures_data = [
    (1, '2024-11-17 08:15'),
    (2, '2024-11-17 08:10'),
    (3, '2024-11-17 08:20'),
    (4, '2024-11-17 08:25'),
    (5, '2024-11-17 08:20'),
    (6, '2024-11-17 13:00'),
    (7, '2024-11-17 12:25'),
    (8, '2024-11-17 12:30'),
    (9, '2024-11-17 15:05'),
    (10, '2024-11-17 15:10'),
    (11, '2024-11-17 15:15'),
    (12, '2024-11-17 15:15'),
    (13, '2024-11-17 20:15'),
    (14, '2024-11-17 20:15')
]

In [0]:
# Define schema for the data
arrival_columns = ['train_id', 'arrival_time']
departure_columns = ['train_id', 'departure_time']

In [0]:
# Create DataFrames
arrivals_df = spark.createDataFrame(arrivals_data, arrival_columns)
departures_df = spark.createDataFrame(departures_data, departure_columns)


In [0]:
# Convert the time strings to timestamps for easier handling
arrivals_df = arrivals_df.withColumn('arrival_time', F.col('arrival_time').cast('timestamp'))
departures_df = departures_df.withColumn('departure_time', F.col('departure_time').cast('timestamp'))

In [0]:
# Add event type (arrival = 1, departure = -1)
arrivals_df = arrivals_df.withColumn('event_type', F.lit(1))
departures_df = departures_df.withColumn('event_type', F.lit(-1))

In [0]:
# Union both DataFrames into one, marking events as either arrival or departure
all_events_df = arrivals_df.select('train_id', 'arrival_time', 'event_type') \
    .withColumnRenamed('arrival_time', 'event_time') \
    .union(departures_df.select('train_id', 'departure_time', 'event_type') \
    .withColumnRenamed('departure_time', 'event_time'))


In [0]:
# Sort events by event_time and prioritize arrivals over departures at the same time
all_events_df = all_events_df.orderBy('event_time', F.col('event_type').desc())


In [0]:
# Use a window function to calculate the running total of platforms needed at each time
window_spec = Window.orderBy('event_time', F.col('event_type').desc())  # Same order as the events


In [0]:
# Calculate running sum of platforms needed
all_events_df = all_events_df.withColumn('platforms_needed', F.sum('event_type').over(window_spec))


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# Find the maximum platforms_needed at any given time
max_platforms = all_events_df.agg(F.max('platforms_needed')).collect()[0][0]


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# Output the result
print(f"The minimum number of platforms required: {max_platforms}")

The minimum number of platforms required: 5
